<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/10_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10 — Deployment

The last module of Part 1.

Nine modules of building agents that run in notebooks or in `adk web`. Module 10 is about turning those agents into **products** — HTTP services that run in production.

ADK gives you three deploy paths:

- **`adk deploy cloud_run`** — one command; deploys to Google Cloud Run. The fast path for anyone who's fine with GCP.
- **Vanilla Docker** — you write a `Dockerfile`, you `docker build`, you deploy wherever containers run — AWS Fargate, Azure Container Apps, fly.io, a Raspberry Pi on your desk. Vendor-neutral.
- **Vertex AI Agent Engine** — the managed, opinionated path. More batteries included (Memory Bank, Agent Identity, built-in eval). Deepest Google lock-in.

This notebook walks through the first two hands-on, and sketches Agent Engine in theory. You will not actually deploy to a cloud — that needs billing and takes 5+ minutes per iteration. What you **will** do: run your agent as a local HTTP API via `adk api_server`, hit it with `curl`, and see exactly what shape your production agent will have once deployed.

**What you'll leave with:**
- The repo layout that makes an agent deployable (one folder, one `agent.py`, `root_agent`).
- `adk api_server` running the agent as a local HTTP service.
- A Dockerfile that runs the same agent on any container platform.
- The one-liner for Cloud Run; the Agent Engine pricing picture.
- When to pick Plugins over Callbacks for production cross-cutting concerns.

**Running cost:** $0 — everything in this notebook is local.

# Setup

In [1]:
!pip install -q google-adk==2.4.0 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


In [2]:
import os, sys, warnings, tempfile, shutil, time, subprocess
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()

# API key (for the running agent, not for deployment itself)
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    except ImportError: pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ Environment ready.")

✅ Environment ready.


# The Deployable Repo Layout

An ADK agent is "deployable" when it lives in a folder with this shape:

```
my_agent/
├── __init__.py          # empty; makes it a package
├── agent.py             # exports `root_agent`
├── requirements.txt     # pinned Python deps
└── .env                 # optional; OPENROUTER_API_KEY etc.
```

`adk api_server`, `adk web`, `adk deploy cloud_run`, `AgentEvaluator.evaluate` all expect this shape. They look for `root_agent` in the `agent.py` (or a module at a path you specify). This is a convention, not a hard requirement — the `adk create` command scaffolds it for you.

Below we write the exact layout in a temp directory.

In [3]:
DEPLOY_DIR = tempfile.mkdtemp(prefix="adk_m10_")
AGENT_DIR = os.path.join(DEPLOY_DIR, "my_weather_agent")
os.makedirs(AGENT_DIR, exist_ok=True)

# __init__.py — make it a package
with open(os.path.join(AGENT_DIR, "__init__.py"), "w") as f:
    f.write("")

# agent.py — the thing the platforms look for
AGENT_CODE = (
    "import os\n"
    "from dotenv import load_dotenv\n"
    "load_dotenv()\n\n"
    "from google.adk.agents import LlmAgent\n"
    "from google.adk.models.lite_llm import LiteLlm\n\n"
    "def get_weather(city: str) -> dict:\n"
    "    'Look up today\u2019s weather for a city.'\n"
    "    db = {\n"
    "        'Bratislava': {'city': 'Bratislava', 'condition': 'Sunny', 'temperature_c': 18},\n"
    "        'Prague': {'city': 'Prague', 'condition': 'Cloudy', 'temperature_c': 14},\n"
    "        'Munich': {'city': 'Munich', 'condition': 'Rainy', 'temperature_c': 11},\n"
    "    }\n"
    "    return db.get(city, {'error': f'No data for {city}'})\n\n"
    "root_agent = LlmAgent(\n"
    "    name='weather_agent',\n"
    "    model=LiteLlm(model='openrouter/google/gemini-2.5-flash-lite'),\n"
    "    description='Reports weather for European cities.',\n"
    "    instruction='Use get_weather. Answer in one sentence.',\n"
    "    tools=[get_weather],\n"
    ")\n"
)

with open(os.path.join(AGENT_DIR, "agent.py"), "w") as f:
    f.write(AGENT_CODE)

# requirements.txt — pinned
with open(os.path.join(AGENT_DIR, "requirements.txt"), "w") as f:
    f.write("google-adk==2.4.0\nlitellm==1.85.7\nopenai==2.45.0\npython-dotenv==1.0.1\ndeprecated==1.2.18\n")

# .env — the key for the running agent
with open(os.path.join(AGENT_DIR, ".env"), "w") as f:
    f.write(f"OPENROUTER_API_KEY={OPENROUTER_API_KEY}\n")

print(f"✅ Deployable layout at {AGENT_DIR}")
print(f"   {sorted(os.listdir(AGENT_DIR))}")

✅ Deployable layout at /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m10_dj3wo65s/my_weather_agent
   ['.env', '__init__.py', 'agent.py', 'requirements.txt']


# Path 1 — Local HTTP Server with `adk api_server`

Before you deploy, see the agent running as an HTTP service. `adk api_server <agents_dir>` starts a FastAPI server hosting all agents in the directory; you can hit it with `curl`.

This is what your production agent actually looks like — an HTTP API with a documented schema. Cloud Run is this same thing, running somewhere else.

In [4]:
# Start adk api_server as a background process
import subprocess, signal

api_proc = subprocess.Popen(
    ["adk", "api_server", DEPLOY_DIR, "--port=8765"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env={**os.environ},
)
time.sleep(5)   # give it time to boot
print(f"✅ adk api_server started (PID {api_proc.pid}), hosting {DEPLOY_DIR}")

✅ adk api_server started (PID 39571), hosting /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m10_dj3wo65s


In [5]:
# Hit it with a real HTTP call.
import json, urllib.request, urllib.error

# The API mirrors Runner.run_async — create a session, then send messages.
BASE = "http://localhost:8765"

def http_post(path, body):
    req = urllib.request.Request(
        BASE + path,
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode())

# Discover what agents are loaded
with urllib.request.urlopen(BASE + "/list-apps") as r:
    apps = json.loads(r.read().decode())
print(f"Available apps: {apps}")

# Create a session
session = http_post(
    "/apps/my_weather_agent/users/tester/sessions/demo-1",
    {},
)
print(f"Session created: id={session.get('id')}")

# Send a message and collect the final response
run_body = {
    "app_name": "my_weather_agent",
    "user_id": "tester",
    "session_id": "demo-1",
    "new_message": {
        "role": "user",
        "parts": [{"text": "What's the weather in Prague?"}],
    },
}
events = http_post("/run", run_body)
print(f"\nGot {len(events)} event(s) back.")
# Find the final text response
for ev in events:
    if ev.get("content", {}).get("parts"):
        for p in ev["content"]["parts"]:
            if p.get("text"):
                print(f"[{ev.get('author','?')}] {p['text'][:150]}")
            if p.get("function_call"):
                print(f"[tool_call] {p['function_call'].get('name')}")

Available apps: ['my_weather_agent']
Session created: id=demo-1



Got 3 event(s) back.
[weather_agent] The weather in Prague is cloudy and 14°C.


This is a production shape. Create a session → POST messages → get events back. The JSON payload format is exactly what the client SDK uses. Swap `http://localhost:8765` for your Cloud Run URL and the same code works.

Stop the server before we move on.

In [6]:
api_proc.send_signal(signal.SIGINT)
time.sleep(1)
api_proc.wait(timeout=5)
print("✅ api_server stopped.")

✅ api_server stopped.


# Path 2 — Vanilla Docker: Deploy Anywhere

`adk api_server` packaged into a container. One Dockerfile, runs on any container platform.

Here's what a production Dockerfile looks like:

In [7]:
DOCKERFILE = '''\
FROM python:3.12-slim

WORKDIR /app

# Install Python deps first (separate layer, better caching)
COPY my_weather_agent/requirements.txt ./requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy the agent code
COPY my_weather_agent/ ./my_weather_agent/

# Cloud Run and most PaaS platforms set $PORT; default to 8080 otherwise.
ENV PORT=8080

# Start the ADK API server hosting the agent folder.
# --host 0.0.0.0 so the container is reachable from outside.
CMD ["sh", "-c", "adk api_server /app --host 0.0.0.0 --port ${PORT}"]
'''

DOCKERFILE_PATH = os.path.join(DEPLOY_DIR, "Dockerfile")
with open(DOCKERFILE_PATH, "w") as f:
    f.write(DOCKERFILE)

print("Dockerfile written:")
print(DOCKERFILE)

Dockerfile written:
FROM python:3.12-slim

WORKDIR /app

# Install Python deps first (separate layer, better caching)
COPY my_weather_agent/requirements.txt ./requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy the agent code
COPY my_weather_agent/ ./my_weather_agent/

# Cloud Run and most PaaS platforms set $PORT; default to 8080 otherwise.
ENV PORT=8080

# Start the ADK API server hosting the agent folder.
# --host 0.0.0.0 so the container is reachable from outside.
CMD ["sh", "-c", "adk api_server /app --host 0.0.0.0 --port ${PORT}"]



The build and run commands, in a shell:

```bash
# From the directory containing the Dockerfile
docker build -t my-weather-agent .

# Run locally, passing the API key as an env var
docker run -p 8080:8080 \
    -e OPENROUTER_API_KEY=sk-or-... \
    my-weather-agent

# Hit it
curl http://localhost:8080/list-apps
```

To deploy elsewhere:

- **AWS Fargate / App Runner:** `aws ecr` push, point a service at the image.
- **Azure Container Apps:** `az acr build`, `az containerapp create`.
- **Fly.io:** `fly launch --dockerfile ./Dockerfile`.
- **Kubernetes:** your usual Helm chart or Kustomize manifest.

The important property: **there's no ADK-specific deployment story for non-Google clouds.** It's regular container deployment. The agent is a FastAPI app; FastAPI apps run everywhere.

# Path 3 — `adk deploy cloud_run`

For Google Cloud, one command does the whole thing:

```bash
adk deploy cloud_run \
    --project YOUR_GCP_PROJECT \
    --region europe-west1 \
    --service_name my-weather-agent \
    my_weather_agent
```

What happens under the hood: ADK generates a Dockerfile, builds the image with Cloud Build, pushes to Artifact Registry, deploys a Cloud Run service. Everything scripted.

**Trade-offs:**

| Pro | Con |
|---|---|
| One command, minutes to prod | Google Cloud only |
| Sensible defaults for an agent workload | Less control than hand-rolling |
| Integrates with other GCP services (Cloud Trace, Cloud Logging) naturally | You pay the Cloud Run runtime margin |

If you're already on GCP and in a hurry — use this. If you're on AWS, Azure, on-prem, or you care about the details of your deployment — the vanilla Docker path above is the one.

# Path 4 — Vertex AI Agent Engine (the Managed Path)

The most opinionated option. Vertex AI Agent Engine is Google's à la carte managed runtime for agents. You deploy; Google runs.

What Agent Engine adds that Cloud Run doesn't:

- **Managed sessions** (alternative to running your own database).
- **Memory Bank** — LLM-distilled long-term memory, auto-consolidation, auto-decay. A real upgrade over what we built in M08.
- **Agent Identity** — per-agent IAM principals with certificate-bound credentials. The strongest enterprise-governance primitive Google ships.
- **Built-in evaluation** and observability — integrates with the Gen AI Evaluation Service from M09.
- **Framework support**: ADK (first-class), LangChain, LangGraph, plus custom-template support for CrewAI, OpenAI Agents SDK, FastAPI.

Pricing (as of April 2026):

- Runtime: $0.0864/vCPU-hour + $0.0090/GiB-hour (same as Cloud Run; no margin at the compute layer).
- Sessions: $0.25/1k events.
- Memory Bank: $0.25/1k memories stored/month; $0.50/1k retrieved (first 1k retrieved/month free).

**When to pick Agent Engine vs DIY Cloud Run:**

- **Use Agent Engine** if you want Memory Bank and Agent Identity without building them yourself. That's the real pitch.
- **Use Cloud Run** if you want portability (the same container runs elsewhere tomorrow), or if you're running at a scale where managed-service margins matter.
- **Use neither** if you're not on Google Cloud.

Agent Engine deployment:

```bash
adk deploy agent_engine \
    --project YOUR_PROJECT \
    --region europe-west1 \
    --staging_bucket gs://your-staging-bucket \
    my_weather_agent
```

# Plugins — the Production Cross-Cutting Concern

Module 07 covered callbacks — per-agent lifecycle hooks. Production often needs the same policy applied to **every agent in the app**: audit logging, per-user rate-limits, token-budget enforcement, org-wide PII redaction.

**Plugins** are the mechanism for that. Registered on the `Runner`, not on individual agents. Every agent managed by the runner goes through the plugin.

Sketch:

```python
from google.adk.plugins import Plugin

class AuditPlugin(Plugin):
    async def before_run(self, context):
        # log every agent invocation
        logger.info("agent_invoked", user=context.user_id, agent=context.agent.name)
    async def after_run(self, context):
        logger.info("agent_finished", duration_ms=context.duration_ms)

runner = Runner(
    agent=root_agent,
    app_name="my_app",
    session_service=sessions,
    plugins=[AuditPlugin()],   # ← applies to EVERY agent the runner touches
)
```

**Rule of thumb:** Callbacks for agent-specific logic; Plugins for app-wide policy. In production, the split is usually:

- Callbacks per specialist for their specific guards.
- Plugins on the runner for org-wide concerns (audit, rate-limits, cost tracking).

Plugins are the newer mechanism; ADK recommends them over app-wide callbacks. If you're refactoring an existing ADK deployment, this is the pattern to migrate toward.

# Production Readiness Checklist

Things to have sorted before you deploy, beyond making the agent work:

1. **Persistence** — `DatabaseSessionService` pointed at managed Postgres (not in-memory, not SQLite on ephemeral disk).
2. **Secrets** — `.env` out of the container; use Cloud Run secrets, AWS Secrets Manager, or similar. Never bake keys into the image.
3. **Callbacks for guardrails** — the blocklist, PII redaction, and mocking-in-tests patterns from M07.
4. **Plugins for observability** — audit logging, cost tracking, rate-limiting.
5. **Eval in CI** — `adk eval` runs on every PR against a curated evalset. Production safety net.
6. **Trace backend** — Cloud Trace, Langfuse, Arize, or your own OpenTelemetry collector. Every event is already instrumented; you just need to receive them.
7. **Health endpoint** — `adk api_server` exposes `/list-apps`; your load balancer needs to know what counts as healthy.
8. **Rate-limit / auth** — ADK doesn't ship auth; put an API gateway in front or run it in an authenticated subnet.

None of these are ADK-specific. They're what any Python service needs. Worth stating explicitly because the ADK demos give you the agent; everything else is yours to build.

# Cleanup

In [8]:
shutil.rmtree(DEPLOY_DIR, ignore_errors=True)
print(f"✅ Cleaned up {DEPLOY_DIR}")

✅ Cleaned up /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m10_dj3wo65s


# Your Turn

1. **Run the local API server manually.** In a terminal: `cp -r` the temp agent dir somewhere durable, then `adk api_server <that_dir>` and use the Swagger UI at `http://localhost:8000/docs` to explore the endpoints.
2. **Build the Docker image.** `docker build -t adk-course-weather .` then `docker run -p 8080:8080 -e OPENROUTER_API_KEY=... adk-course-weather`. Hit the running container with `curl`.
3. **Pick your deployment target.** If you use GCP: actually run `adk deploy cloud_run`. If AWS/Azure/fly.io: deploy the container. If neither: run the container on your laptop and point a frontend at it.
4. **Write a plugin.** Sketch an `AuditPlugin` that logs every agent invocation's user_id + agent name + duration. Register it on the Runner. Verify the logs fire.

# Part 1 Wrap

Ten modules. From "what is an agent" to "how do I ship one."

What you can build now:
- Agents with tools (function, OpenAPI, MCP, agent-tool) — M02
- Persistent state with scope prefixes — M03, M08
- Vendor-neutral model swapping — M04
- Compositions: Sequential, Parallel, Loop, and LLM-driven routing — M05, M06
- Guardrails, caches, PII redaction via callbacks — M07
- Long-term memory via `MemoryService` — M08
- Automated eval with trajectory + LLM-as-judge — M09
- Deployable as an HTTP service anywhere — this module

All on whichever model you want — Claude, GPT, Gemini, a locally-hosted open-weight — because the LiteLLM wrapper made the model a configuration, not a dependency.

# Next up — M11: Gemini unlocks (Part 2 begins)

Part 2 shifts gears. We switch to native Gemini via the `google-genai` SDK and explore the features ADK + Gemini do that nothing else does. Google Search grounding with inline citations. Long-context windows with context caching (90% discount on cached tokens). Thinking budgets that trade latency for reasoning quality. And in M13, the Live API voice agent — the single most-differentiated Gemini-only capability as of 2026.

Switch your `.env` to include `GOOGLE_API_KEY` (free tier at aistudio.google.com is enough for M11-M13). See you there.